# Dask Out-of-Core S3 Validation

This notebook validates Dask out-of-core processing with S3 storage.

**Note**: This notebook is read-only from ConfigMap. To edit:
```python
import shutil
shutil.copy('/home/jovyan/sample-notebooks/Dask_S3_Validation.ipynb', '/home/jovyan/')
```

In [ ]:
# Imports
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
import dask.dataframe as dd
from dask.distributed import Client

import holoviews as hv
hv.extension('bokeh')

# AWS credentials are provided via:
# - IAM role (when running in AWS with proper instance profile)
# - Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN)
# - ~/.aws/credentials file
# No hardcoded credentials - uses boto3's credential chain
print("Using AWS credential chain (IAM role / env vars / credentials file)")

## 1. Connect to Dask Cluster

In [ ]:
# Connect to remote scheduler or create local
scheduler_address = os.getenv('DASK_SCHEDULER_ADDRESS')
if scheduler_address:
    print(f"Connecting to: {scheduler_address}")
    client = Client(scheduler_address)
else:
    print("Creating local cluster")
    client = Client(n_workers=2)

print(f"Dashboard: {client.dashboard_link}")
client

## 2. Configure S3 Access

In [ ]:
# S3 configuration (uses IAM role in AWS)
s3 = s3fs.S3FileSystem()  # Uses IAM role automatically

# Data bucket
S3_BUCKET = os.getenv('OTEL_DATA_PATH', 's3://cybersec-dask-data/otel/').replace('s3://', '').rstrip('/')
bucket_name = S3_BUCKET.split('/')[0]
data_prefix = '/'.join(S3_BUCKET.split('/')[1:]) or 'otel'

print(f"Bucket: {bucket_name}")
print(f"Prefix: {data_prefix}")

# Test access
contents = s3.ls(bucket_name)
print(f"\nBucket contents: {contents}")

## 3. Generate Large Synthetic Dataset (Idempotent)

We generate **10 million spans** across **100 partitions** to demonstrate out-of-core processing.
Each span includes additional attributes to increase memory footprint (~500 bytes/row in memory).

**Dataset Size:**
- Disk: ~2-3 GB (Parquet compressed)
- Memory: ~5-8 GB (fully expanded)
- This exceeds individual worker memory, forcing streaming/out-of-core processing

In [ ]:
# Configuration for large dataset
TOTAL_SPANS = 10_000_000      # 10 million spans
NUM_PARTITIONS = 100          # 100 parquet files
SPANS_PER_PARTITION = TOTAL_SPANS // NUM_PARTITIONS  # 100K per partition
NUM_SERVICES = 20             # More realistic service count

validation_path = f"{bucket_name}/{data_prefix}/validation-large"

# Check if data already exists (idempotent)
try:
    existing_files = s3.glob(f"{validation_path}/**/*.parquet")
    if existing_files:
        total_size = sum(s3.info(f)['size'] for f in existing_files)
        print(f"✓ Data exists: {len(existing_files)} files ({total_size/1024/1024:.1f} MB)")
        print(f"  Path: s3://{validation_path}/")
        print("  Skipping generation (idempotent)")
        GENERATE_DATA = False
    else:
        print("No existing data found, will generate")
        GENERATE_DATA = True
except Exception as e:
    print(f"No existing data: {e}")
    GENERATE_DATA = True

In [ ]:
def generate_spans(n_spans=100000, n_services=20):
    """Generate synthetic span data with rich attributes."""
    services = [f"service-{i:02d}" for i in range(n_services)]
    operations = [
        'GET /api/v1/users', 'POST /api/v1/orders', 'GET /health',
        'PUT /api/v1/users/{id}', 'DELETE /api/v1/orders/{id}',
        'db.query.select', 'db.query.insert', 'db.query.update',
        'cache.get', 'cache.set', 'cache.delete',
        'queue.publish', 'queue.consume',
        'grpc.client.call', 'grpc.server.handle',
    ]
    http_methods = ['GET', 'POST', 'PUT', 'DELETE', 'PATCH']
    status_codes_http = [200, 201, 204, 400, 401, 403, 404, 500, 502, 503]

    now = datetime.now(timezone.utc)
    base_ts = int(now.timestamp() * 1e9)

    # Generate random data
    n = n_spans
    service_idx = np.random.randint(0, n_services, n)

    return pd.DataFrame({
        # Core span fields
        'trace_id': [uuid.uuid4().hex for _ in range(n)],
        'span_id': [uuid.uuid4().hex[:16] for _ in range(n)],
        'parent_span_id': np.where(
            np.random.random(n) > 0.3,
            [uuid.uuid4().hex[:16] for _ in range(n)],
            ''
        ),
        'service_name': np.array(services)[service_idx],
        'operation_name': np.random.choice(operations, n),

        # Timing
        'start_time_unix_nano': base_ts - np.random.randint(0, 86400 * 1e9, n).astype(np.int64),
        'duration_ns': np.random.exponential(scale=50_000_000, size=n).astype(np.int64),

        # Status
        'status_code': np.random.choice(['OK', 'ERROR', 'UNSET'], n, p=[0.92, 0.05, 0.03]),
        'http_status_code': np.random.choice(status_codes_http, n, p=[0.5, 0.1, 0.05, 0.1, 0.05, 0.05, 0.05, 0.05, 0.025, 0.025]),
        'http_method': np.random.choice(http_methods, n, p=[0.5, 0.25, 0.1, 0.1, 0.05]),

        # Additional attributes for richer data
        'user_id': [f"user-{np.random.randint(1, 10000):05d}" for _ in range(n)],
        'request_id': [uuid.uuid4().hex for _ in range(n)],
        'host': np.array([f"{services[i]}-{np.random.randint(1,10)}.cluster.local" for i in service_idx]),
        'db_statement': np.where(
            np.random.random(n) > 0.7,
            [f"SELECT * FROM table_{np.random.randint(1,100)} WHERE id = {np.random.randint(1,1000000)}" for _ in range(n)],
            ''
        ),
        'error_message': np.where(
            np.random.random(n) > 0.95,
            np.random.choice(['Connection timeout', 'Rate limit exceeded', 'Internal error', 'Not found', 'Permission denied'], n),
            ''
        ),
    })

if GENERATE_DATA:
    print(f"Generating {TOTAL_SPANS:,} spans across {NUM_PARTITIONS} partitions...")
    print(f"  {SPANS_PER_PARTITION:,} spans per partition")
    print(f"  {NUM_SERVICES} services")
    print()

    for partition in range(NUM_PARTITIONS):
        df = generate_spans(n_spans=SPANS_PER_PARTITION, n_services=NUM_SERVICES)

        # Convert to PyArrow table
        table = pa.Table.from_pandas(df, preserve_index=False)

        # Write to S3
        output_path = f"s3://{validation_path}/partition_{partition:03d}.parquet"
        pq.write_table(table, output_path, filesystem=s3, compression='snappy')
        
        if (partition + 1) % 10 == 0:
            print(f"  Written {partition + 1}/{NUM_PARTITIONS} partitions...")

    # Print final stats
    files = s3.glob(f"{validation_path}/*.parquet")
    total_size = sum(s3.info(f)['size'] for f in files)
    print(f"\n✓ Data generation complete!")
    print(f"  Files: {len(files)}")
    print(f"  Total size: {total_size/1024/1024/1024:.2f} GB")
else:
    print("Using existing data")

## 4. Load Data with Dask (Lazy)

In [ ]:
# Load as Dask DataFrame (lazy - no data loaded yet)
ddf = dd.read_parquet(
    f"s3://{validation_path}/*.parquet",
    storage_options={'anon': False},  # Use IAM role
)

print(f"Partitions: {ddf.npartitions}")
print(f"Columns: {list(ddf.columns)}")
print(f"Expected rows: {TOTAL_SPANS:,}")
print(f"\n⚡ This is LAZY - no data loaded yet!")
print("   Data will only be fetched when .compute() is called")
ddf

## 5. Compute Aggregations (Distributed)

In [ ]:
%%time
# This triggers distributed computation across all workers
# Watch the Dask dashboard to see tasks being distributed!
total_spans = len(ddf)
print(f"✓ Total spans: {total_spans:,}")
print(f"  Processing {total_spans:,} rows across {ddf.npartitions} partitions")

In [ ]:
%%time
# Multi-dimensional aggregation - watch the Dask dashboard!
# This demonstrates streaming aggregation: data flows through workers
# without loading the entire dataset into memory

service_stats = ddf.groupby('service_name').agg({
    'span_id': 'count',
    'duration_ns': ['mean', 'max', 'min', 'std'],
    'http_status_code': ['mean'],
}).compute()

service_stats.columns = ['count', 'mean_duration_ns', 'max_duration_ns', 'min_duration_ns', 'std_duration_ns', 'avg_http_status']
service_stats['mean_duration_ms'] = service_stats['mean_duration_ns'] / 1e6
service_stats['max_duration_ms'] = service_stats['max_duration_ns'] / 1e6
service_stats = service_stats.sort_values('count', ascending=False)

print(f"✓ Aggregated {total_spans:,} spans into {len(service_stats)} service groups")
service_stats[['count', 'mean_duration_ms', 'max_duration_ms', 'std_duration_ns']]

In [ ]:
%%time
# Error rate by service
error_rate = ddf.groupby('service_name')['status_code'].apply(
    lambda x: (x == 'ERROR').sum() / len(x) * 100,
    meta=('error_rate', 'float64')
).compute()

print("Error rate by service (%)")
error_rate

## 6. Visualizations with HoloViews

In [ ]:
# Sample for visualization (compute a subset)
sample_df = ddf.sample(frac=0.1).compute()
print(f"Sample size: {len(sample_df):,} spans")

In [ ]:
# Duration distribution by service
sample_df['duration_ms'] = sample_df['duration_ns'] / 1e6

hv.BoxWhisker(sample_df, kdims='service_name', vdims='duration_ms').opts(
    width=800, height=400,
    title='Latency Distribution by Service',
    ylabel='Duration (ms)',
    box_fill_color=hv.dim('service_name').categorize(dict(zip(
        sample_df['service_name'].unique(),
        hv.Cycle('Category10').values
    )))
)

In [ ]:
# Span count by service (bar chart)
counts = sample_df.groupby('service_name').size().reset_index(name='count')

hv.Bars(counts, kdims='service_name', vdims='count').opts(
    width=600, height=400,
    title='Span Count by Service',
    color='service_name',
    cmap='Category10',
    xrotation=45,
)

In [ ]:
# Status distribution
status_counts = sample_df.groupby(['service_name', 'status_code']).size().reset_index(name='count')

hv.Bars(status_counts, kdims=['service_name', 'status_code'], vdims='count').opts(
    width=800, height=400,
    title='Status Code Distribution by Service',
    xrotation=45,
    color='status_code',
    cmap={'OK': 'green', 'ERROR': 'red', 'UNSET': 'gray'},
)

## 7. Out-of-Core Stress Test

This section demonstrates that Dask processes data **larger than worker memory** using streaming aggregation.
Each worker processes partitions sequentially, computing partial results and releasing memory.

In [ ]:
import time

def show_worker_memory(label=""):
    """Display worker memory utilization."""
    info = client.scheduler_info()
    print(f"\n{'='*60}")
    print(f"Worker Memory Utilization {label}")
    print(f"{'='*60}")
    
    total_used = 0
    total_limit = 0
    
    for worker_id, worker in sorted(info['workers'].items()):
        mem_used = worker.get('metrics', {}).get('memory', 0)
        mem_limit = worker['memory_limit']
        pct = (mem_used / mem_limit * 100) if mem_limit else 0
        worker_short = worker_id.split('/')[-1][:25]
        bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
        print(f"  {worker_short:25} [{bar}] {mem_used/1e9:.2f}GB / {mem_limit/1e9:.2f}GB ({pct:5.1f}%)")
        total_used += mem_used
        total_limit += mem_limit
    
    print(f"\n  TOTAL: {total_used/1e9:.2f}GB / {total_limit/1e9:.2f}GB ({total_used/total_limit*100:.1f}%)")
    return total_used, total_limit

# Show baseline memory
show_worker_memory("(baseline)");

In [ ]:
%%time
# Complex multi-key aggregation - stresses out-of-core processing
# This groups by multiple columns, requiring more intermediate state

print("Running complex aggregation across 10M spans...")
print("Watch the Dask dashboard - you'll see tasks streaming through workers\n")

complex_stats = ddf.groupby(['service_name', 'operation_name', 'http_method']).agg({
    'span_id': 'count',
    'duration_ns': ['mean', 'max', 'std'],
    'http_status_code': lambda x: (x >= 400).sum(),  # Error count
}).compute()

complex_stats.columns = ['count', 'mean_duration_ns', 'max_duration_ns', 'std_duration_ns', 'error_count']
complex_stats['mean_duration_ms'] = complex_stats['mean_duration_ns'] / 1e6
complex_stats['error_rate_pct'] = complex_stats['error_count'] / complex_stats['count'] * 100
complex_stats = complex_stats.sort_values('count', ascending=False)

print(f"✓ Created {len(complex_stats)} distinct groups from {total_spans:,} spans")
print(f"  Unique service/operation/method combinations: {len(complex_stats)}")
complex_stats.head(15)[['count', 'mean_duration_ms', 'error_rate_pct']]

In [ ]:
# Check memory AFTER complex aggregation
# Key insight: memory should still be bounded even after processing 10M rows
show_worker_memory("(after complex aggregation)")

print("\n" + "="*60)
print("OUT-OF-CORE VERIFICATION")
print("="*60)
print("""
✓ If worker memory stayed below 80%, out-of-core processing worked!
  
  How it works:
  1. Dask reads partitions lazily from S3 (one at a time per task)
  2. Each partition is processed and partial results accumulated
  3. Memory is released after each partition is processed
  4. Only final aggregated results stay in memory
  
  This allows processing datasets MUCH larger than available RAM.
""")

## 8. Cleanup

In [ ]:
# Optionally delete test data
# Uncomment to clean up:
# s3.rm(validation_path, recursive=True)
# print(f"Deleted: {validation_path}")

In [ ]:
# Close client
# client.close()

---

## Validation Summary

This notebook validated **Dask out-of-core processing** at scale:

| Metric | Value |
|--------|-------|
| Total Spans | 10,000,000 |
| Partitions | 100 |
| Services | 20 |
| Disk Size | ~2-3 GB |
| Memory Footprint | ~5-8 GB (expanded) |

### Capabilities Demonstrated

1. **Dask Cluster Connection** - Connected to distributed scheduler with multiple workers
2. **S3 Access via IAM** - Read/write to S3 using AWS IAM role credentials
3. **Idempotent Data Generation** - Skips regeneration if data already exists
4. **Lazy Loading** - 10M rows loaded only when `.compute()` is called
5. **Distributed Aggregations** - GroupBy operations computed in parallel across workers
6. **Out-of-Core Processing** - Memory stays bounded even for large datasets
7. **HoloViews Visualizations** - Interactive plots render correctly in JupyterLab

### Key Insight

Dask's out-of-core processing allows analyzing datasets **larger than cluster memory** by:
- Streaming partitions through workers one at a time
- Computing partial aggregations on each partition
- Combining partial results into final output
- Releasing memory after each partition is processed